# télos AR: Autoregressive Baseline Training Suites
This notebook executes the training pipeline for **Autoregressive (AR) Causal Baselines**.

Key features:
- Additive lower-triangular causal attention mask
- Standard next-token cross-entropy loss
- Exact parameter & hyperparameter parity with MDLM and UNDLM suites.

In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "ar").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from ar.model.mlx_components import MLXCausalTransformer
from ar.training.trainer import TelosMLXARTrainer

def run_ar_training_step(config_path, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING AR BASELINE TRAINING RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXCausalTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if resume_from:
        print(f"  [Resume] Loading weights from {resume_from}")
        model.load_weights(resume_from, strict=False)
    
    trainer = TelosMLXARTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED AR RUN: " + str(config_path) + "\n")

In [ ]:
# PIPELINE DEFINITION: AR Baseline 25M 1:35 Run
start_time = time.time()

run_ar_training_step("configs/masked/25m/phase_b_25m_1to35_mlx.yaml")

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"AR BASELINE 25M 1:35 RUN COMPLETED IN {total_elapsed:.2f} HOURS!")
print("=" * 85)